## Selección del modelo final

El modelo `bagging_rf_model.pkl` fue seleccionado como artefacto final del Sprint 4 porque corresponde a un ensamble basado en Random Forest, alineado con el PB-14 de técnicas avanzadas.

Como parte del rol Experiment Tracker, el modelo fue copiado y persistido como:

`models/final_model.pkl`

Luego se validó que el modelo final puede cargarse correctamente con `joblib` y generar predicciones sobre una muestra del conjunto de prueba.

Además, el resultado final fue registrado en:

`models/experiments_log.csv`

con sus métricas, ruta del modelo, parámetros relevantes y marca de modelo seleccionado.

In [1]:
import joblib
from pathlib import Path

MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")

bagging_path = MODELS_DIR / "bagging_rf_model.pkl"
# Asumimos que el archivo del umbral sigue existiendo en la carpeta
threshold_path = MODELS_DIR / "bagging_rf_threshold.pkl" 
final_path = MODELS_DIR / "final_model.pkl"

print("Existe bagging_rf_model.pkl:", bagging_path.exists())

# 1. Cargamos el modelo y el umbral específico
model_objeto = joblib.load(bagging_path)
umbral_objeto = joblib.load(threshold_path)

# 2. Creamos un contenedor (diccionario) con ambos elementos
artefacto_final = {
    "model": model_objeto,
    "threshold": umbral_objeto
}

# 3. Guardamos el contenedor unificado usando joblib
joblib.dump(artefacto_final, final_path)

print("¡Artefacto final unificado con su umbral específico!")
print("Guardado en:", final_path)

Existe bagging_rf_model.pkl: True
¡Artefacto final unificado con su umbral específico!
Guardado en: ../models/final_model.pkl


In [2]:
import pandas as pd
import joblib
from pathlib import Path

# 1. Configuración de rutas
MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

# 2. Carga de datos de prueba
test_df = pd.read_csv(DATA_DIR / "processed" / "test_original.csv")
X_test = test_df.drop("y", axis=1)
y_test = test_df["y"]

# 3. Carga del artefacto unificado (Diccionario)
artefacto = joblib.load(MODELS_DIR / "final_model.pkl")

# 4. Desempaquetamos el modelo puro y tu umbral específico
modelo_puro = artefacto["model"]
umbral_optimo = artefacto["threshold"]

# 5. Tomamos las primeras 5 muestras y las convertimos a NumPy para evitar el error de columnas
X_test_sample = X_test.iloc[:5].to_numpy()

# 6. REEMPLAZO DE PREDICCIÓN: Usamos predict_proba + el umbral óptimo
probabilidades = modelo_puro.predict_proba(X_test_sample)[:, 1]
preds_con_umbral = (probabilidades >= umbral_optimo).astype(int)

# 7. Impresión de resultados
print(f"Umbral óptimo aplicado: {umbral_optimo:.4f}")
print("Predicciones del modelo final (con umbral ajustado):")
print(preds_con_umbral)

/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but BaggingClassifier was fitted with feature names
  warnings.warn(


Umbral óptimo aplicado: 0.2038
Predicciones del modelo final (con umbral ajustado):
[0 0 0 0 0]


In [3]:
import os
import sys
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

# =====================================================================
# IMPORTAMOS LAS LIBRERÍAS DE MLFLOW
# =====================================================================
import mlflow
import mlflow.sklearn

sys.path.append(os.path.abspath(".."))
from src.tuning import log_sprint4_result  # Solo importamos el logger

MODELS_DIR = Path("models") if Path("models").exists() else Path("../models")
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

# Cargar datos de prueba
test_df = pd.read_csv(DATA_DIR / "processed" / "test_original.csv")
X_test = test_df.drop("y", axis=1)
y_test = test_df["y"]

# 1. Cargamos el artefacto completo (Diccionario: modelo + umbral)
artefacto = joblib.load(MODELS_DIR / "final_model.pkl")
modelo_puro = artefacto["model"]
umbral_optimo = artefacto["threshold"]

# 2. CALCULO MANUAL CON EL UMBRAL ÓPTIMO (Evitando el error de feature_names_in_)
X_test_np = X_test.to_numpy() if hasattr(X_test, "columns") else X_test

# Obtenemos probabilidades de la clase positiva
probabilidades = modelo_puro.predict_proba(X_test_np)[:, 1]
# Aplicamos tu umbral específico optimizado
preds_con_umbral = (probabilidades >= umbral_optimo).astype(int)

# 3. Construimos las métricas reales (CORREGIDO EL CRUCE DE VARIABLES)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, preds_con_umbral, average="binary"
)
auc_roc = roc_auc_score(y_test, probabilidades)

# Calculamos Accuracy global para guardarlo en MLflow
accuracy = (preds_con_umbral == y_test).mean()

metrics = {
    "accuracy": accuracy,
    "f1": f1,  # Corregido
    "recall": recall,
    "precision": precision,  # Corregido
    "auc_roc": auc_roc,
}

# Generamos reporte y matriz para visualizarlos en el notebook
report = classification_report(y_test, preds_con_umbral)
cm = confusion_matrix(y_test, preds_con_umbral)

# =====================================================================
# SECCIÓN AUTOMÁTICA DE MLFLOW 
# =====================================================================
# Fuerza a MLflow a guardar la carpeta 'mlruns' en la raíz corporativa del proyecto
mlflow.set_tracking_uri("file:../mlruns")

# Configuramos el nombre del experimento en la UI
mlflow.set_experiment("Final_Validation")

# Iniciamos la corrida oficial en la bitácora gráfica de MLflow
with mlflow.start_run(run_name="Sprint_Final_Model_Prod"):

    # Registramos los parámetros clave del modelo y el umbral aplicado
    mlflow.log_param("model_type", "Bagging Random Forest")
    mlflow.log_param("source_model", "bagging_rf_model.pkl")
    mlflow.log_param("final_artifact", "final_model.pkl")
    mlflow.log_param("applied_threshold", float(umbral_optimo))
    mlflow.log_param("stage", "PB-15 Final Validation")

    # Registramos automáticamente las métricas del test set en la UI
    mlflow.log_metric("accuracy", metrics["accuracy"])
    mlflow.log_metric("precision", metrics["precision"])
    mlflow.log_metric("recall", metrics["recall"])
    mlflow.log_metric("f1_score", metrics["f1"])
    mlflow.log_metric("roc_auc", metrics["auc_roc"])

    # Guardamos el modelo binario puro dentro de MLflow para gobernanza
    mlflow.sklearn.log_model(modelo_puro, artifact_path="models")

print("¡Evidencia registrada con éxito en MLflow UI!")

# =====================================================================
# REGISTRO EN EL ARCHIVO CSV HISTÓRICO
# =====================================================================
result = log_sprint4_result(
    model="Bagging Random Forest",
    experiment_type="Final Model Validation",
    metrics=metrics,
    params={
        "source_model": "bagging_rf_model.pkl",
        "final_artifact": "final_model.pkl",
        "applied_threshold": umbral_optimo,
    },
    model_path=str(MODELS_DIR / "final_model.pkl"),
    stage="PB-15 Final Validation",
    selected=True,
    notes=f"Modelo final con umbral optimizado ({umbral_optimo:.4f}) validado en test set y trackeado en MLflow.",
    log_path=str(MODELS_DIR / "experiments_log.csv"),
)

print(
    "\nMétricas finales logradas con el umbral óptimo (Calculadas Manualmente):"
)
print(metrics)

print("\nReporte de Clasificación Completo:")
print(report)

/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but BaggingClassifier was fitted with feature names
  warnings.warn(
/Users/davisjoel/miniforge3/envs/environment/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/29 01:07:59 INFO mlflow.tracking.fluent: Experiment with name 'Final_Validation' does not exist. Creating a new experiment.
2026/05/29 01:08:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 01:08:00 WARNING mlflow.sklearn: Saving

¡Evidencia registrada con éxito en MLflow UI!

Métricas finales logradas con el umbral óptimo (Calculadas Manualmente):
{'accuracy': np.float64(0.6975953364100073), 'f1': 0.358908341915551, 'recall': 0.7518878101402373, 'precision': 0.23571187013865405, 'auc_roc': 0.7982705328002628}

Reporte de Clasificación Completo:
              precision    recall  f1-score   support

           0       0.96      0.69      0.80      7307
           1       0.24      0.75      0.36       927

    accuracy                           0.70      8234
   macro avg       0.60      0.72      0.58      8234
weighted avg       0.88      0.70      0.75      8234

